# 10. Colab CUDA: официальный distill 5s

Путь к полноценной генерации на NVIDIA. Включайте тяжёлые ячейки отдельно; все учебные лабы 00–08 также можно выполнять в Colab без GPU.


### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


## Официальный inference_examples_t2v.ipynb: читаем перед запуском

В reference сохранён оригинальный notebook того же pinned commit. В нём показаны SFT, noCFG, Distil, 5s/10s и Pro; для нашей цели нужна только **Lite Distil 5s**. Не выполняйте весь notebook: он последовательно создаёт разные большие pipelines.

Оригинал использует `os.chdir("..")`, подразумевая запуск из examples/. Наш курс использует абсолютные пути. Загрузка весов и установка зависимостей в оригинальном notebook не организованы: это примеры вызова уже подготовленной среды, а не самостоятельный Colab quickstart.


In [ ]:
import json
example=json.loads((SOURCE/'examples/inference_examples_t2v.ipynb').read_text())
distill_code=next(''.join(c['source']) for c in example['cells'] if c['cell_type']=='code' and 'k5_lite_t2v_5s_distil_sd.yaml' in ''.join(c['source']))
print(distill_code)
assert 'get_video_pipeline' in distill_code and 'cuda:0' in distill_code



### Какие defaults скрывает короткая ячейка

| Параметр | В оригинале | В исследовательском запуске ниже |
|---|---|---|
| Backbone | distill config | Тот же distill 5s, strict checkpoint |
| Steps / guidance | Берутся из config: 16 / 1 | Задаются явно: 16 / 1 |
| Seed | Не задан, выбирается случайно | 42 для повторяемого сравнения |
| Prompt expansion | Pipeline default True | False: исследуем точный исходный prompt |
| Scheduler scale | Pipeline default 10 | 5 явно; отдельный опыт сравнивает 5/10 |
| Attention engine | auto | SDPA: без обязательной установки FlashAttention |
| Offload | По умолчанию False | True: экономим VRAM |
| Placement | Все модели cuda:0 | DiT/VAE CUDA; Qwen/CLIP CPU в baseline |

Первое воспроизведение можно сделать со scale=10, чтобы сохранить original default; scale=5 — значение YAML и наш явный исследовательский baseline. Различие фиксируйте в отчёте, не считайте два результата одинаковой конфигурацией.

Веса Qwen на CPU требуют порядка 13 GiB только для BF16 параметров и дополнительную память. Успех GPU проверки не гарантирует достаточную системную RAM. Вариант GPU Qwen + offload/NF4 — отдельная CUDA оптимизация, а не обязательная исходная зависимость.

### Mac или Colab?

Оригинальный pipeline вызывает CUDA set_device, CUDA generator/autocast и CUDA-aware VAE tiling. Замены строк cuda:0 на mps недостаточно. Mac notebook 09 использует отдельный experimental adapter; small MPS FP32/BF16 проверены, pretrained 5s пока не проверен. Для первого full-scale baseline выбираем CUDA, затем сравниваем с Mac port.

Colab GPU model и RAM не гарантированы. Проверяйте BF16 и ресурсы runtime; T4 не поддерживает выбранную native BF16 ветвь аппаратно и не является базовым маршрутом данного курса. Предпочтительный первый опыт — A100 и high-memory runtime, если доступны. [Colab resource FAQ](https://research.google.com/colaboratory/faq.html#resource-limits).


In [ ]:
for relative,needle in [('kandinsky/utils.py','torch.cuda.set_device'),('kandinsky/generation_utils.py','g = torch.Generator(device="cuda")'),('kandinsky/models/vae.py','free_mem = torch.cuda.mem_get_info')]:
    print('CUDA dependency:',relative)
    source_excerpt(relative,needle,5)



### 1. Выбираем runtime

Runtime → Change runtime type → GPU. Colab не гарантирует модель GPU и время доступа. T4 поддерживает FP16, но не аппаратный BF16; официальный код широко использует BF16. Для этого маршрута нужен BF16-capable GPU, предпочтительно A100/L4. Мало VRAM: сначала conditioning на CPU, offload, VAE tiling; успешный запуск не гарантируется.


In [ ]:
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),'VRAM GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)
    print('BF16:',torch.cuda.is_bf16_supported())
else: print('Здесь проверяется только подготовка; нужен GPU runtime для инференса.')


### 2. Подготовка официальной среды

Используем уже сохранённую pinned копию кода reference/kandinsky-5. В notebook путь до кода добавляется явно. Дополнительные зависимости нужны официальному pipeline: torchvision, peft, av, opencv. Ячейка установки отключена по умолчанию. При замене PyTorch перезапустите runtime перед импортами.


In [ ]:
INSTALL_CUDA_EXTRAS=False
if INSTALL_CUDA_EXTRAS:
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','torchvision==0.23.0','peft>=0.17,<0.18','av','opencv-python-headless'])
else: print('Дополнительные CUDA-зависимости не устанавливались.')


### 3. Только distill checkpoint и shared компоненты

Не вызываем download_models.py без --models: он скачивает несколько моделей. Наш downloader фильтрует компоненты и фиксирует revisions. Shared weights не уменьшаются от перехода SFT→distill.


In [ ]:
DOWNLOAD_WEIGHTS=False
if DOWNLOAD_WEIGHTS:
    from labkit.components import download_component
    for name in ['qwen','clip','dit','vae']: print(name,download_component(name))
else: print('Скачивание весов пропущено.')


### 4. Конфигурация с абсолютными путями

В официальный YAML вносим только пути к выбранным файлам. Hunyuan build_vae сам добавляет subfolder=vae, поэтому checkpoint_path указывает на корень weights/vae. Текстовые checkpoint_path указывают непосредственно на dirs qwen/clip.


In [ ]:
from omegaconf import OmegaConf
conf=OmegaConf.load(SOURCE/'configs/k5_lite_t2v_5s_distil_sd.yaml')
conf.model.checkpoint_path=str(ROOT/'weights/dit/model/kandinsky5lite_t2v_distilled16steps_5s.safetensors')
conf.model.vae.checkpoint_path=str(ROOT/'weights/vae')
conf.model.text_embedder.qwen.checkpoint_path=str(ROOT/'weights/qwen')
conf.model.text_embedder.clip.checkpoint_path=str(ROOT/'weights/clip')
config_path=ROOT/'artifacts/colab_distill.yaml'
OmegaConf.save(conf,config_path)
print(OmegaConf.to_yaml(conf))


### 5. Как создавать pipeline в этой версии

У pinned revision exported factory называется get_video_pipeline(mode="t2v"). README upstream показывает get_T2V_pipeline, но в текущем __init__.py такого alias нет. Поэтому используем функцию, реально существующую в сохранённом коде. SDPA выбран явно, MagCache выключен для чистого baseline.


In [ ]:
RUN_PIPELINE=False
if RUN_PIPELINE:
    assert torch.cuda.is_available(),'Нужен CUDA GPU'
    assert torch.cuda.is_bf16_supported(),'Этот official BF16 путь не предназначен для T4'
    if str(SOURCE) not in sys.path: sys.path.insert(0,str(SOURCE))
    from kandinsky import get_video_pipeline
    pipe=get_video_pipeline(device_map={'dit':torch.device('cuda:0'),'vae':torch.device('cuda:0'),'text_embedder':torch.device('cpu')},conf_path=str(config_path),offload=True,attention_engine='sdpa',text_token_padding=True,quantized_qwen=False,magcache=False,mode='t2v')
else: print('Официальный pipeline не создавался.')


### 6. Полная генерация и измерение

Перед каждым запуском переместим энкодеры на CPU: с offload официальный pipeline в конце может вернуть их на configured text device. seed/steps/guidance/scale задаём явно. Первый замер включает часть подготовки и compile; второй после прогрева сравним отдельно. Вызов сохраняет 5s видео и tensor результата.


In [ ]:
RUN_GENERATION=False
if RUN_GENERATION:
    assert RUN_PIPELINE,'Сначала создайте pipeline'
    import time
    from labkit.common import save_json
    prompt='A cat in a red hat slowly turns its head. Static camera, warm daylight.'
    pipe.text_embedder.to('cpu')
    torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize(); start=time.perf_counter()
    images=pipe(text=prompt,time_length=5,width=768,height=512,seed=42,num_steps=16,guidance_weight=1.,scheduler_scale=5.,expand_prompts=False,save_path=str(ROOT/'artifacts/colab_distill_5s.mp4'))
    torch.cuda.synchronize(); elapsed=time.perf_counter()-start
    report={'elapsed_seconds':elapsed,'peak_allocated_GiB':torch.cuda.max_memory_allocated()/2**30,'GPU':torch.cuda.get_device_name(0),'seed':42,'steps':16,'NFE':16,'scale':5.,'guidance':1.,'prompt':prompt,'source_revision':REVISION}
    save_json(ROOT/'artifacts/colab_distill_5s.json',report); print(report)
    from IPython.display import Video,display
    display(Video(str(ROOT/'artifacts/colab_distill_5s.mp4'),embed=True))
else: print('Полная CUDA генерация НЕ выполнялась.')


### 7. Если памяти не хватает

Qwen на CPU экономит VRAM, но требует RAM Colab и может быть медленным. На сервере с CUDA можно исследовать NF4 Qwen отдельно; это дополнительная зависимость bitsandbytes и отдельный эксперимент. 16 steps снижает вычисления, но число latent tokens сохраняется. Не скрывайте OOM автоматической заменой checkpoint.


In [ ]:
from labkit.common import device_info
print(device_info())
print('CPU Qwen BF16 ~13 GiB только округлённые веса; загрузка и hidden states требуют запас.')


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
